In [71]:
import pandas as pd
import numpy as np
!pip install thefuzz
from thefuzz import fuzz
import itertools

In [72]:
data = {
    "order_id":   [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "customer":   ["Alice Johnson", "Bob Smith", "alice johnson",
                   "BOB SMITH", "Charlie D.", "Charlie D",
                   "Diana Prince", "Diana Prince"],
    "email":      ["alice@mail.com", "bob@mail.com", "alice@mail.com",
                   "bob@mail.com", "charlie@mail.com", "charlie@mail.com",
                   "diana@mail.com", "diana@mail.com"],
    "amount":     [500, 300, 500, 300, 150, 150, 200, 275],
    "date":       ["2024-01-01", "2024-01-02", "2024-01-01", "2024-01-02",
                   "2024-01-03", "2024-01-03", "2024-01-04", "2024-01-05"]
}
df = pd.DataFrame(data)

## inspecting the original data frame

In [73]:
df

,order_id,customer,email,amount,date
0,1001,Alice Johnson,alice@mail.com,500,2024-01-01
1,1002,Bob Smith,bob@mail.com,300,2024-01-02
2,1003,alice johnson,alice@mail.com,500,2024-01-01
3,1004,BOB SMITH,bob@mail.com,300,2024-01-02
4,1005,Charlie D.,charlie@mail.com,150,2024-01-03
5,1006,Charlie D,charlie@mail.com,150,2024-01-03
6,1007,Diana Prince,diana@mail.com,200,2024-01-04
7,1008,Diana Prince,diana@mail.com,275,2024-01-05


## duplicate pandas detected

In [74]:
df.duplicated().sum() #duplicates detected by pandas

np.int64(0)

## normalizing the customer and email:

In [75]:
df["customer_normalized"] = df["customer"].str.lower().str.strip()

print(df[["customer","customer_normalized"]])

        customer customer_normalized
0  Alice Johnson       alice johnson
1      Bob Smith           bob smith
2  alice johnson       alice johnson
3      BOB SMITH           bob smith
4     Charlie D.          charlie d.
5      Charlie D           charlie d
6   Diana Prince        diana prince
7   Diana Prince        diana prince


In [76]:
df["email_normalized"] = df["email"].str.lower().str.strip() # not much changed, but a good practice to normalise it

df[["customer","customer_normalized","email","email_normalized"]]


,customer,customer_normalized,email,email_normalized
0,Alice Johnson,alice johnson,alice@mail.com,alice@mail.com
1,Bob Smith,bob smith,bob@mail.com,bob@mail.com
2,alice johnson,alice johnson,alice@mail.com,alice@mail.com
3,BOB SMITH,bob smith,bob@mail.com,bob@mail.com
4,Charlie D.,charlie d.,charlie@mail.com,charlie@mail.com
5,Charlie D,charlie d,charlie@mail.com,charlie@mail.com
6,Diana Prince,diana prince,diana@mail.com,diana@mail.com
7,Diana Prince,diana prince,diana@mail.com,diana@mail.com


## Defining a key column- as it can be seen that no single column qualify to be a key column, so i'll create a composite key

In [77]:
df

,order_id,customer,email,amount,date,customer_normalized,email_normalized
0,1001,Alice Johnson,alice@mail.com,500,2024-01-01,alice johnson,alice@mail.com
1,1002,Bob Smith,bob@mail.com,300,2024-01-02,bob smith,bob@mail.com
2,1003,alice johnson,alice@mail.com,500,2024-01-01,alice johnson,alice@mail.com
3,1004,BOB SMITH,bob@mail.com,300,2024-01-02,bob smith,bob@mail.com
4,1005,Charlie D.,charlie@mail.com,150,2024-01-03,charlie d.,charlie@mail.com
5,1006,Charlie D,charlie@mail.com,150,2024-01-03,charlie d,charlie@mail.com
6,1007,Diana Prince,diana@mail.com,200,2024-01-04,diana prince,diana@mail.com
7,1008,Diana Prince,diana@mail.com,275,2024-01-05,diana prince,diana@mail.com


In [78]:
df["composite_key"] = df["date"] + "||" + df["email"]
df

,order_id,customer,email,amount,date,customer_normalized,email_normalized,composite_key
0,1001,Alice Johnson,alice@mail.com,500,2024-01-01,alice johnson,alice@mail.com,2024-01-01||alice@mail.com
1,1002,Bob Smith,bob@mail.com,300,2024-01-02,bob smith,bob@mail.com,2024-01-02||bob@mail.com
2,1003,alice johnson,alice@mail.com,500,2024-01-01,alice johnson,alice@mail.com,2024-01-01||alice@mail.com
3,1004,BOB SMITH,bob@mail.com,300,2024-01-02,bob smith,bob@mail.com,2024-01-02||bob@mail.com
4,1005,Charlie D.,charlie@mail.com,150,2024-01-03,charlie d.,charlie@mail.com,2024-01-03||charlie@mail.com
5,1006,Charlie D,charlie@mail.com,150,2024-01-03,charlie d,charlie@mail.com,2024-01-03||charlie@mail.com
6,1007,Diana Prince,diana@mail.com,200,2024-01-04,diana prince,diana@mail.com,2024-01-04||diana@mail.com
7,1008,Diana Prince,diana@mail.com,275,2024-01-05,diana prince,diana@mail.com,2024-01-05||diana@mail.com


## now it's better now, pandas can detect the duplicates

In [79]:
df[df.duplicated(subset=["composite_key"], keep=False)]

,order_id,customer,email,amount,date,customer_normalized,email_normalized,composite_key
0,1001,Alice Johnson,alice@mail.com,500,2024-01-01,alice johnson,alice@mail.com,2024-01-01||alice@mail.com
1,1002,Bob Smith,bob@mail.com,300,2024-01-02,bob smith,bob@mail.com,2024-01-02||bob@mail.com
2,1003,alice johnson,alice@mail.com,500,2024-01-01,alice johnson,alice@mail.com,2024-01-01||alice@mail.com
3,1004,BOB SMITH,bob@mail.com,300,2024-01-02,bob smith,bob@mail.com,2024-01-02||bob@mail.com
4,1005,Charlie D.,charlie@mail.com,150,2024-01-03,charlie d.,charlie@mail.com,2024-01-03||charlie@mail.com
5,1006,Charlie D,charlie@mail.com,150,2024-01-03,charlie d,charlie@mail.com,2024-01-03||charlie@mail.com


## The above rows are duplicate rows, These are needed to be dropped.

In [80]:
dupes = df[df.duplicated(subset=['composite_key'], keep=False)]
print("The Following Duplicates will be collapsed: ")
dupes

The Following Duplicates will be collapsed: 


,order_id,customer,email,amount,date,customer_normalized,email_normalized,composite_key
0,1001,Alice Johnson,alice@mail.com,500,2024-01-01,alice johnson,alice@mail.com,2024-01-01||alice@mail.com
1,1002,Bob Smith,bob@mail.com,300,2024-01-02,bob smith,bob@mail.com,2024-01-02||bob@mail.com
2,1003,alice johnson,alice@mail.com,500,2024-01-01,alice johnson,alice@mail.com,2024-01-01||alice@mail.com
3,1004,BOB SMITH,bob@mail.com,300,2024-01-02,bob smith,bob@mail.com,2024-01-02||bob@mail.com
4,1005,Charlie D.,charlie@mail.com,150,2024-01-03,charlie d.,charlie@mail.com,2024-01-03||charlie@mail.com
5,1006,Charlie D,charlie@mail.com,150,2024-01-03,charlie d,charlie@mail.com,2024-01-03||charlie@mail.com


In [81]:
# as the above output shows the duplicate, I will remove them.
df_deduped = df.drop_duplicates(subset=['composite_key'],keep='first')
df_deduped

,order_id,customer,email,amount,date,customer_normalized,email_normalized,composite_key
0,1001,Alice Johnson,alice@mail.com,500,2024-01-01,alice johnson,alice@mail.com,2024-01-01||alice@mail.com
1,1002,Bob Smith,bob@mail.com,300,2024-01-02,bob smith,bob@mail.com,2024-01-02||bob@mail.com
4,1005,Charlie D.,charlie@mail.com,150,2024-01-03,charlie d.,charlie@mail.com,2024-01-03||charlie@mail.com
6,1007,Diana Prince,diana@mail.com,200,2024-01-04,diana prince,diana@mail.com,2024-01-04||diana@mail.com
7,1008,Diana Prince,diana@mail.com,275,2024-01-05,diana prince,diana@mail.com,2024-01-05||diana@mail.com


## resetting the index

In [82]:
df_deduped = df_deduped.reset_index(drop=True)

In [83]:
df_deduped

,order_id,customer,email,amount,date,customer_normalized,email_normalized,composite_key
0,1001,Alice Johnson,alice@mail.com,500,2024-01-01,alice johnson,alice@mail.com,2024-01-01||alice@mail.com
1,1002,Bob Smith,bob@mail.com,300,2024-01-02,bob smith,bob@mail.com,2024-01-02||bob@mail.com
2,1005,Charlie D.,charlie@mail.com,150,2024-01-03,charlie d.,charlie@mail.com,2024-01-03||charlie@mail.com
3,1007,Diana Prince,diana@mail.com,200,2024-01-04,diana prince,diana@mail.com,2024-01-04||diana@mail.com
4,1008,Diana Prince,diana@mail.com,275,2024-01-05,diana prince,diana@mail.com,2024-01-05||diana@mail.com


In [84]:
df_deduped["customer"] = df_deduped["customer_normalized"].str.title()
df_deduped["email"] = df_deduped["email_normalized"]
df_deduped

,order_id,customer,email,amount,date,customer_normalized,email_normalized,composite_key
0,1001,Alice Johnson,alice@mail.com,500,2024-01-01,alice johnson,alice@mail.com,2024-01-01||alice@mail.com
1,1002,Bob Smith,bob@mail.com,300,2024-01-02,bob smith,bob@mail.com,2024-01-02||bob@mail.com
2,1005,Charlie D.,charlie@mail.com,150,2024-01-03,charlie d.,charlie@mail.com,2024-01-03||charlie@mail.com
3,1007,Diana Prince,diana@mail.com,200,2024-01-04,diana prince,diana@mail.com,2024-01-04||diana@mail.com
4,1008,Diana Prince,diana@mail.com,275,2024-01-05,diana prince,diana@mail.com,2024-01-05||diana@mail.com


## dropping the helper columns

In [85]:
df_deduped = df_deduped.drop(columns=['customer_normalized','email_normalized','composite_key'])
df_deduped

,order_id,customer,email,amount,date
0,1001,Alice Johnson,alice@mail.com,500,2024-01-01
1,1002,Bob Smith,bob@mail.com,300,2024-01-02
2,1005,Charlie D.,charlie@mail.com,150,2024-01-03
3,1007,Diana Prince,diana@mail.com,200,2024-01-04
4,1008,Diana Prince,diana@mail.com,275,2024-01-05


In [86]:
df_deduped.duplicated().sum()

np.int64(0)

In [87]:
print("Final clean DataFrame: ")
df_deduped

Final clean DataFrame: 


,order_id,customer,email,amount,date
0,1001,Alice Johnson,alice@mail.com,500,2024-01-01
1,1002,Bob Smith,bob@mail.com,300,2024-01-02
2,1005,Charlie D.,charlie@mail.com,150,2024-01-03
3,1007,Diana Prince,diana@mail.com,200,2024-01-04
4,1008,Diana Prince,diana@mail.com,275,2024-01-05


In [88]:
print("Shape of the DataFrame: ")
df_deduped.shape

Shape of the DataFrame: 


(5, 5)

In [89]:
# end of the File